# Part A: Transit accessibility per commune

Builds `transit_access.csv`: one row per (commune, site) pair in the survey data.

* **Origin:** the commune's main public-transport stop (railway station if it has one, otherwise the central bus stop), taken from the official timetable stop database (transport.opendata.ch). Fallback: geocoded postcode centroid, flagged in `origin_type`.
* **Travel times and timetables:** Google Maps Platform (Routes API, transit and driving).

**Setup:** enable *Geocoding API* and *Routes API* in Google Cloud, create an API key, and save it alone in a text file named `api_key.txt` next to this notebook. Do not submit that file.

**Run first with `LIMIT = 3`**, check the output, then set `LIMIT = None`.

In [1]:
import os, json, time, hashlib, getpass
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import unicodedata, re

In [12]:
# ---------------- Configuration ----------------

DATA_PATH = "parking_data_clean.csv"
TZ = ZoneInfo("Europe/Zurich")

# A normal weekday, outside school holidays, BEFORE the 13 Dec 2026 timetable change.
# Check the Vaud and Valais school calendars before fixing it.
QUERY_DATE = date(2026, 11, 3)

# Replace with the real employer addresses if you know them.
DESTINATIONS = {
    "Lausanne": "Bâtiment CE – 3.316, Station 1, CH-1015 Lausanne, Switzerland",
    "Sion": "Route des Ronquos 86, 1950 Sion, Switzerland",
}

# Key is read from api_key.txt (one line, no quotes). Keep that file out of anything you submit.
API_KEY = Path("api_key.txt").read_text().strip()

STEP_MIN = 10          # spacing of departure queries between 06:00 and 09:00
LIMIT = None              # number of (commune, site) pairs to run; None = all
CACHE_DIR = Path("api_cache")
CACHE_DIR.mkdir(exist_ok=True)

ROUTES_URL = "https://routes.googleapis.com/directions/v2:computeRoutes"
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"
FIELD_MASK = "routes.duration,routes.distanceMeters,routes.legs.steps.travelMode,routes.legs.steps.transitDetails"

In [13]:
# ---------------- Helpers: time, caching, API calls ----------------
def at(hhmm):
    h, m = map(int, hhmm.split(":"))
    return datetime(QUERY_DATE.year, QUERY_DATE.month, QUERY_DATE.day, h, m, tzinfo=TZ)

def _key(*parts):
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()

def _cached(key, fetch):
    f = CACHE_DIR / f"{key}.json"
    if f.exists():
        return json.loads(f.read_text())
    out = fetch()
    f.write_text(json.dumps(out))
    time.sleep(0.05)   # be gentle with the API
    return out

def compute_route(origin, dest, mode, when=None, kind="departure"):
    """origin/dest = (lat, lng). kind = 'departure' or 'arrival' (arrival only valid for TRANSIT)."""
    def ll(p):
        return {"location": {"latLng": {"latitude": p[0], "longitude": p[1]}}}
    payload = {"origin": ll(origin), "destination": ll(dest), "travelMode": mode,
               "languageCode": "en", "units": "METRIC"}
    if mode == "DRIVE":
        payload["routingPreference"] = "TRAFFIC_AWARE"
    if when is not None:
        payload[f"{kind}Time"] = when.isoformat()
    def fetch():
        r = requests.post(ROUTES_URL, json=payload,
                          headers={"X-Goog-Api-Key": API_KEY, "X-Goog-FieldMask": FIELD_MASK})
        if r.status_code != 200:
            raise RuntimeError(f"Routes API {r.status_code}: {r.text[:300]}")
        return r.json()
    j = _cached(_key("route", payload), fetch)
    routes = j.get("routes") or []
    return routes[0] if routes else None

def minutes(route):
    return int(route["duration"].rstrip("s")) / 60 if route else np.nan

def transit_steps(route):
    if not route:
        return []
    steps = [s for leg in route.get("legs", []) for s in leg.get("steps", [])]
    return [s["transitDetails"] for s in steps if "transitDetails" in s]

def ts(s):
    return pd.Timestamp(s).tz_convert(TZ)

In [14]:
# ---------------- The three indicators for one (origin, destination) ----------------
def indicators(origin, dest):
    out = {}

    # 1) car time arriving ~08:00. DRIVE has no arrivalTime, so do two passes.
    first = compute_route(origin, dest, "DRIVE", at("07:30"))
    d1 = minutes(first)
    if not np.isnan(d1):
        dep = (at("08:00") - timedelta(minutes=d1)).replace(second=0, microsecond=0)
        second = compute_route(origin, dest, "DRIVE", dep)
        out["car_min"] = minutes(second)
        out["car_km"] = second["distanceMeters"] / 1000
    else:
        out["car_min"], out["car_km"] = np.nan, np.nan

    # 1b) transit time arriving by 08:00
    tr = compute_route(origin, dest, "TRANSIT", at("08:00"), kind="arrival")
    st = transit_steps(tr)
    if not st:
        tr = None
    else:
        arr = ts(st[-1]["stopDetails"]["arrivalTime"])
        if arr.date() != QUERY_DATE or arr > at("08:00"):
            tr = None
    out["transit_min_0800"] = minutes(tr)
    out["transit_ratio"] = out["transit_min_0800"] / out["car_min"]

    # 2) distinct transit departures (first boarding) between 06:00 and 09:00
    start, end = at("06:00"), at("09:00")
    seen, t = set(), start
    while t <= end:
        r = compute_route(origin, dest, "TRANSIT", t, kind="departure")
        steps = transit_steps(r)
        if steps:
            d = steps[0]
            dep_ts = ts(d["stopDetails"]["departureTime"])
            line = d.get("transitLine", {}).get("nameShort") or d.get("transitLine", {}).get("name", "?")
            if start <= dep_ts <= end:
                seen.add((dep_ts.isoformat(), line))
        t += pd.Timedelta(minutes=STEP_MIN)
    out["departures_0600_0900"] = len(seen)

    # 3) first feasible arrival: leave at 04:00, take the first itinerary's arrival time
    early = compute_route(origin, dest, "TRANSIT", at("04:00"), kind="departure")
    steps = transit_steps(early)
    if steps:
        arr = ts(steps[-1]["stopDetails"]["arrivalTime"])
        out["first_arrival_time"] = arr.strftime("%H:%M")
        out["first_arrival_hours"] = arr.hour + arr.minute / 60
    else:
        out["first_arrival_time"], out["first_arrival_hours"] = None, np.nan
    return out

In [15]:
# ---------------- Unique (commune, site) pairs from the survey data ----------------
df = pd.read_csv(DATA_PATH)

pairs = (df.groupby(["home_commune", "site"])
           .agg(postcode=("home_postcode", lambda s: s.mode().iloc[0]),
                n_postcodes=("home_postcode", "nunique"),
                n_employees=("employee_id", "nunique"),
                survey_median_km=("distance_km", "median"))
           .reset_index())
print(len(pairs), "commune-site pairs;", df["home_commune"].nunique(), "communes")
pairs.head()

39 commune-site pairs; 39 communes


,home_commune,site,postcode,n_postcodes,n_employees,survey_median_km
0,Arbaz,Sion,1974,1,6,11.20
1,Ardon,Sion,1957,1,11,13.70
2,Ayent,Sion,1966,1,15,12.00
3,Bottens,Lausanne,1041,1,15,14.20
4,Bramois,Sion,1967,1,16,5.35


In [16]:
# ---------------- Geocoding (used for destinations and the centroid fallback) ----------------
def geocode_info(address):
    address = " ".join(address.split())
    f = CACHE_DIR / f"{_key('geocode', address)}.json"
    if f.exists():
        j = json.loads(f.read_text())
    else:
        r = requests.get(GEOCODE_URL, params={"address": address, "region": "ch",
                         "components": "country:CH", "key": API_KEY})
        r.raise_for_status()
        j = r.json()
        if j.get("status") == "OK":              # only cache successes
            f.write_text(json.dumps(j))
    if j.get("status") != "OK":
        raise ValueError(f"Geocoding failed for {address!r}: {j.get('status')} - {j.get('error_message')}")
    res = j["results"][0]
    loc = res["geometry"]["location"]
    return loc["lat"], loc["lng"], res.get("formatted_address", ""), res.get("types", [])

def geocode(address):
    return geocode_info(address)[:2]

# ---------------- Origins: official public-transport stops (transport.opendata.ch) ----------------
LOC_URL = "https://transport.opendata.ch/v1/locations"

def _n(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[\s\-']+", " ", s).strip().lower()

def stop_candidates(commune):
    f = CACHE_DIR / f"{_key('opendata_loc', commune)}.json"
    if f.exists():
        return json.loads(f.read_text())
    r = requests.get(LOC_URL, params={"query": commune, "type": "station"}, timeout=30)
    r.raise_for_status()
    st = r.json().get("stations", [])
    f.write_text(json.dumps(st)); time.sleep(0.2)
    return st

PREFER = ("centre", "village", "poste", "gare", "eglise", "commune")

def pick_origin(commune, postcode):
    """Return ((lat, lng), origin_type, stop_name). In this API, coordinate x = latitude, y = longitude."""
    cn = _n(commune)
    cands = []
    for s in stop_candidates(commune):
        c = s.get("coordinate") or {}
        if c.get("x") is None or c.get("y") is None:
            continue
        nm = _n(s["name"])
        if nm == cn:
            cands.append((0, s, "rail_station"))            # bare commune name = main station
        elif nm.startswith(cn + ","):
            rank = 1 if any(w in nm for w in PREFER) else 2
            cands.append((rank, s, "bus_stop"))
    if cands:
        _, s, typ = sorted(cands, key=lambda t: t[0])[0]
        return (s["coordinate"]["x"], s["coordinate"]["y"]), typ, s["name"]
    return geocode(f"{postcode} {commune}, Switzerland"), "centroid", ""

In [17]:
# ---------------- Geocode destinations and pick origins ----------------
dest_xy = {site: geocode(addr) for site, addr in DESTINATIONS.items()}
print(dest_xy)

todo = pairs if LIMIT is None else pairs.head(LIMIT)
origins, origin_meta = {}, {}
for _, p in todo.iterrows():
    key = (p.home_commune, p.postcode)
    if key not in origins:
        origins[key], typ, name = pick_origin(p.home_commune, p.postcode)
        origin_meta[key] = {"origin_type": typ, "origin_stop": name}

# CHECK THIS TABLE: each stop name should match its commune; count how many fell back to 'centroid'
pd.DataFrame(origin_meta).T

{'Lausanne': (46.5206137, 6.5691269), 'Sion': (46.2245767, 7.359878299999999)}


,,origin_type,origin_stop
Arbaz,1974,bus_stop,"Arbaz, poste"
Ardon,1957,rail_station,Ardon
Ayent,1966,centroid,
Bottens,1041,bus_stop,"Bottens, croisée"
Bramois,1967,bus_stop,"Bramois, Pont de Bramois"
Bussigny,1030,rail_station,Bussigny
Chavannes-Près-Renens,1022,centroid,
Cheseaux-Sur-Lausanne,1033,centroid,
Conthey,1964,bus_stop,"Conthey, centre commercial"
Corcelles-Le-Jorat,1082,bus_stop,"Corcelles-le-Jorat, village"


In [18]:
# ---------------- Run the queries (cached on disk, safe to re-run) ----------------
rows = []
for i, p in todo.iterrows():
    key = (p.home_commune, p.postcode)
    o = origins[key]
    rec = p.to_dict()
    rec.update(origin_meta[key])
    rec.update({"origin_lat": o[0], "origin_lng": o[1],
                "destination_address": DESTINATIONS[p.site]})
    try:
        rec.update(indicators(o, dest_xy[p.site]))
        rec["error"] = ""
    except Exception as e:
        rec["error"] = str(e)[:200]
    rows.append(rec)
    print(f"{p.home_commune} -> {p.site} [{rec['origin_type']}]: ratio={rec.get('transit_ratio')}, "
          f"departures={rec.get('departures_0600_0900')}, first_arrival={rec.get('first_arrival_time')} {rec['error']}")

res = pd.DataFrame(rows)
res.head()

Arbaz -> Sion [bus_stop]: ratio=1.7979197622585439, departures=4, first_arrival=05:51 
Ardon -> Sion [rail_station]: ratio=1.7140600315955763, departures=6, first_arrival=04:45 
Ayent -> Sion [centroid]: ratio=2.379878577623591, departures=5, first_arrival=06:30 
Bottens -> Lausanne [bus_stop]: ratio=2.121176470588235, departures=11, first_arrival=06:32 
Bramois -> Sion [bus_stop]: ratio=2.644490644490644, departures=12, first_arrival=None 
Bussigny -> Lausanne [rail_station]: ratio=1.7536585365853659, departures=15, first_arrival=05:32 
Chavannes-Près-Renens -> Lausanne [centroid]: ratio=2.5770392749244713, departures=17, first_arrival=None 
Cheseaux-Sur-Lausanne -> Lausanne [centroid]: ratio=2.25662100456621, departures=13, first_arrival=05:42 
Conthey -> Sion [bus_stop]: ratio=2.611683848797251, departures=10, first_arrival=04:45 
Corcelles-Le-Jorat -> Lausanne [bus_stop]: ratio=2.485743380855397, departures=9, first_arrival=05:52 
Crissier -> Lausanne [bus_stop]: ratio=2.3472022955

,home_commune,site,postcode,n_postcodes,n_employees,survey_median_km,origin_type,origin_stop,origin_lat,origin_lng,destination_address,car_min,car_km,transit_min_0800,transit_ratio,departures_0600_0900,first_arrival_time,first_arrival_hours,error
0,Arbaz,Sion,1974,1,6,11.20,bus_stop,"Arbaz, poste",46.272639,7.382858,"Route des Ronquos 86, 1950 Sion, Switzerland",22.433333,12.385,40.333333,1.797920,4,05:51,5.850000,
1,Ardon,Sion,1957,1,11,13.70,rail_station,Ardon,46.208595,7.270946,"Route des Ronquos 86, 1950 Sion, Switzerland",10.550000,8.917,18.083333,1.714060,6,04:45,4.750000,
2,Ayent,Sion,1966,1,15,12.00,centroid,,46.280897,7.413291,"Route des Ronquos 86, 1950 Sion, Switzerland",19.216667,10.576,45.733333,2.379879,5,06:30,6.500000,
3,Bottens,Lausanne,1041,1,15,14.20,bus_stop,"Bottens, croisée",46.616782,6.660038,"Bâtiment CE – 3.316, Station 1, CH-1015 Lausan...",28.333333,21.515,60.100000,2.121176,11,06:32,6.533333,
4,Bramois,Sion,1967,1,16,5.35,bus_stop,"Bramois, Pont de Bramois",46.229026,7.401300,"Route des Ronquos 86, 1950 Sion, Switzerland",8.016667,3.540,21.200000,2.644491,12,NaN,NaN,


In [19]:
# ---------------- Composite score and ordered categories ----------------
r = res.copy()
r["no_transit_route"] = r["transit_min_0800"].isna()

# Missing indicators are coded at the worst observed value, so communes without
# a feasible route end up in the lowest category.
r["transit_ratio_f"] = r["transit_ratio"].fillna(r["transit_ratio"].max())
r["first_arrival_f"] = r["first_arrival_hours"].fillna(r["first_arrival_hours"].max())
r["departures_f"] = r["departures_0600_0900"].fillna(0)

def z(s):
    return (s - s.mean()) / s.std(ddof=0)

# Higher = better access, so flip the sign of the ratio and of the arrival time.
r["z_ratio"] = -z(r["transit_ratio_f"])
r["z_departures"] = z(r["departures_f"])
r["z_first_arrival"] = -z(r["first_arrival_f"])
r["transit_score"] = r[["z_ratio", "z_departures", "z_first_arrival"]].mean(axis=1)

# Terciles at (commune, site) level; rank first so ties cannot break qcut.
r["transit_group"] = pd.qcut(r["transit_score"].rank(method="first"), 3,
                             labels=["Low", "Medium", "High"])

# Coding notes
r["notes"] = ""
def add_note(mask, text):
    r.loc[mask, "notes"] = r.loc[mask, "notes"] + text + "; "
add_note(r["no_transit_route"], "No usable 08:00 transit route: ratio coded at worst value")
add_note(r["departures_0600_0900"].fillna(0) == 0, "No departures found 06:00-09:00")
add_note(r["first_arrival_hours"].isna(), "No early arrival found: coded at worst value")
add_note(r["origin_type"] == "centroid", "Origin = postcode centroid (no stop found)")
add_note(r["n_postcodes"] > 1, "Commune spans several postcodes (modal one used)")

r["source"] = "Google Maps Platform (Routes API, Geocoding API); stop locations from transport.opendata.ch"
r["query_date"] = QUERY_DATE.isoformat()
r.sort_values("transit_score", ascending=False).head(10)

,home_commune,site,postcode,n_postcodes,n_employees,survey_median_km,origin_type,origin_stop,origin_lat,origin_lng,...,first_arrival_f,departures_f,z_ratio,z_departures,z_first_arrival,transit_score,transit_group,notes,source,query_date
26,Pully,Lausanne,1009,1,49,4.80,rail_station,Pully,46.510926,6.659472,...,5.700000,13,2.113301,0.551029,0.779458,1.147929,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
20,Lutry,Lausanne,1095,1,46,7.90,rail_station,Lutry,46.503785,6.690607,...,5.700000,12,2.047004,0.367353,0.779458,1.064605,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
23,Morges,Lausanne,1110,1,43,12.40,rail_station,Morges,46.511099,6.493966,...,5.533333,18,0.481277,1.469411,0.996903,0.982530,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
5,Bussigny,Lausanne,1030,1,43,9.60,rail_station,Bussigny,46.547588,6.551827,...,5.533333,15,0.975317,0.918382,0.996903,0.963534,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
18,Lausanne,Lausanne,1004,1,138,2.90,rail_station,Lausanne,46.516793,6.629091,...,5.533333,18,0.361055,1.469411,0.996903,0.942456,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
13,Epalinges,Lausanne,1066,1,29,6.30,bus_stop,"Epalinges, Croisettes",46.543096,6.661547,...,5.866667,18,0.480652,1.469411,0.562012,0.837358,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
1,Ardon,Sion,1957,1,11,13.70,rail_station,Ardon,46.208595,7.270946,...,4.750000,6,1.053962,-0.734706,2.018895,0.779384,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
22,Montpreveyres,Lausanne,1081,1,10,15.85,bus_stop,"Montpreveyres, village",46.582304,6.740877,...,5.866667,12,0.595013,0.367353,0.562012,0.508126,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
28,Romanel-Sur-Lausanne,Lausanne,1032,1,20,5.60,rail_station,Romanel-sur-Lausanne,46.562469,6.603174,...,5.700000,12,0.238855,0.367353,0.779458,0.461888,High,,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03
7,Cheseaux-Sur-Lausanne,Lausanne,1033,1,21,8.40,centroid,,46.585452,6.604771,...,5.700000,13,-0.023596,0.551029,0.779458,0.435630,High,Origin = postcode centroid (no stop found);,"Google Maps Platform (Routes API, Geocoding AP...",2026-11-03


In [20]:
# ---------------- Validation ----------------
print("Corr(Google driving km, survey median distance_km):",
      r[["car_km", "survey_median_km"]].corr().iloc[0, 1].round(3))
r["km_ratio"] = r["car_km"] / r["survey_median_km"]
print("Pairs with Google/survey distance ratio outside [0.7, 1.4] (check geocoding):")
print(r.loc[(r["km_ratio"] < 0.7) | (r["km_ratio"] > 1.4),
            ["home_commune", "site", "car_km", "survey_median_km"]])
print(r.groupby("transit_group", observed=True)[["transit_ratio", "departures_0600_0900", "first_arrival_hours"]].mean())
print(r["origin_type"].value_counts())

Corr(Google driving km, survey median distance_km): 0.763
Pairs with Google/survey distance ratio outside [0.7, 1.4] (check geocoding):
             home_commune      site  car_km  survey_median_km
1                   Ardon      Sion   8.917             13.70
3                 Bottens  Lausanne  21.515             14.20
4                 Bramois      Sion   3.540              5.35
6   Chavannes-Près-Renens  Lausanne   2.293              7.20
9      Corcelles-Le-Jorat  Lausanne  27.359             18.60
11                   Cugy  Lausanne  17.585             10.40
12               Ecublens  Lausanne   2.657              8.20
13              Epalinges  Lausanne  16.214              6.30
15            Froideville  Lausanne  21.548             14.60
17              Heremence      Sion  12.955             19.40
18               Lausanne  Lausanne   5.292              2.90
19   Le Mont-Sur-Lausanne  Lausanne  15.254              5.70
20                  Lutry  Lausanne  22.816              7

In [21]:
# ---------------- Export ----------------
cols = ["home_commune", "site", "postcode", "n_employees", "origin_type", "origin_stop",
        "origin_lat", "origin_lng", "destination_address", "car_min", "car_km",
        "transit_min_0800", "transit_ratio", "departures_0600_0900",
        "first_arrival_time", "first_arrival_hours",
        "transit_score", "transit_group", "no_transit_route",
        "source", "query_date", "notes", "error"]
r[cols].to_csv("transit_access.csv", index=False)
print("saved", len(r), "rows")

saved 39 rows
